# Paper 27 backbone on YOLO26n: LapConv (Laplacian edge enhancement)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 27:** **LapConv** at layer 1 (the second conv of the backbone); every other backbone layer is YOLO26n base |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free; Paper 27 used an OBB head, not used here) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.7 MB/s eta 0:00:00


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so lapconv_backbone.py can be imported

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 11358.91it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Paper 27 backbone: LapConv

**What LapConv does** (Paper 27, Eq. 8–12):
1. A **fixed** 3×3 Laplacian kernel [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]] is applied as a depthwise conv → edge map E.
2. The edge map is added back with one learnable scalar: X' = X + α · E.
3. X' goes through the normal conv + BN + SiLU of the layer.

**Where it sits in YOLO26n** (Paper 27: "the second convolutional layer of the backbone").

| Layer | Base YOLO26n | Paper 27 backbone | Output |
|---|---|---|---|
| 1 | Conv 3×3, stride 2 (16 → 32) | **LapConv**: Laplacian edge map on the 16-channel input, X + α·E, then the same Conv 3×3 s2 | P2/4 (160 × 160) |

The output shape is unchanged, so no layer index changes and the neck connections stay the same.

**Pretrained weights (exact warm start).** `yolo26n.pt` is loaded first, then layer 1 is wrapped: the pretrained conv
is kept, the Laplacian kernel is fixed (not trained) and **α starts at 0**, so at the start the model gives exactly the
same output as the base model. Only α (1 parameter) is learned. Paper 27 initialised α = 1.0; with a pretrained conv that
would change its input at the start, so 0 is used here (change `ALPHA_INIT` in the file if you want 1.0).

**Cost.** One fixed 3×3 depthwise filter on a 16-channel 320 × 320 map: almost no extra FLOPs and no extra weights
except α. It exports to NCNN as a normal depthwise convolution.

In [4]:
%%writefile lapconv_backbone.py
"""Paper 27 backbone for YOLO26n: LapConv at layer 1 (fixed Laplacian depthwise edge map, X + alpha * E, then the original Conv)."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

LAP_LAYERS = (1,)
ALPHA_INIT = 0.0  # 0 = exact warm start from yolo26n.pt (Paper 27 used 1.0)
LAPLACIAN = torch.tensor([[-1.0, -1.0, -1.0], [-1.0, 8.0, -1.0], [-1.0, -1.0, -1.0]])


class LapConv(nn.Module):
    """Laplacian edge enhancement in front of an existing Conv: Y = Conv(X + alpha * DWConv_Laplacian(X))."""

    def __init__(self, conv, alpha=ALPHA_INIT):
        super().__init__()
        c = conv.conv.in_channels
        self.register_buffer("lap", LAPLACIAN.view(1, 1, 3, 3).repeat(c, 1, 1, 1))  # fixed, not trained
        self.alpha = nn.Parameter(torch.tensor([float(alpha)]))
        self.conv = conv  # the pretrained Conv (conv + BN + SiLU)
        self.c = c

    def forward(self, x):
        e = F.conv2d(x, self.lap, padding=1, groups=self.c)
        return self.conv(x + self.alpha * e)


def add_lapconv(model, layers=LAP_LAYERS):
    """Wrap the Conv at the given backbone layers with LapConv (the Conv keeps its weights). Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv), f"layer {i} is {type(old).__name__}, not Conv"
        new = LapConv(old).to(next(old.parameters()).device)
        new.i, new.f, new.type = old.i, old.f, "LapConv"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class LapConvTrainer(DetectionTrainer):
    """Loads the pretrained weights first, then wraps layer 1 with LapConv (alpha starts at ALPHA_INIT)."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_lapconv(model)
        if RANK in {-1, 0}:
            print(f"Paper 27 backbone: LapConv at layers {LAP_LAYERS} (alpha init {ALPHA_INIT})")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


Writing lapconv_backbone.py


In [5]:
import importlib, lapconv_backbone
importlib.reload(lapconv_backbone)
from lapconv_backbone import add_lapconv, LapConvTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 27 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
lap = add_lapconv(deepcopy(base)).eval()
with torch.no_grad():
    x = torch.rand(1, 3, 640, 640)
    yb, yl = base(x), lap(x)
    yb = yb[0] if isinstance(yb, (list, tuple)) else yb
    yl = yl[0] if isinstance(yl, (list, tuple)) else yl
    print(f"warm start (alpha = {lap.model[1].alpha.item()}): max |base - LapConv| over the whole model = {(yb - yl).abs().max():.2e} (should be 0)")
    lap.model[1].alpha.fill_(1.0)
    _ = lap(x)  # forward check with an active edge branch
    lap.model[1].alpha.fill_(lapconv_backbone.ALPHA_INIT)
print("layer 1:", type(lap.model[1]).__name__, "| trainable extra parameters:", lap.model[1].alpha.numel())
for tag, m in (("base", base), ("Paper 27 backbone", lap)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

Overriding model.yaml nc=80 with nc=27
warm start (alpha = 0.0): max |base - LapConv| over the whole model = 0.00e+00 (should be 0)
layer 1: LapConv | trainable extra parameters: 1
base               Params 2.3801 M | GFLOPs 5.338
Paper 27 backbone  Params 2.3801 M | GFLOPs 5.338


## 4. Training

Only the Paper 27 backbone model is trained.

In [6]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("LapConv Backbone (Paper 27)", "bb_lapconv"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=LapConvTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== LapConv Backbone (Paper 27) (bb_lapconv) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.49G      1.342       2.23    0.01816         14        640: 100% ━━━━━━━━━━━━ 482/482 4.9it/s 1:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.2it/s 2.5s
                   all        550        557      0.745      0.792      0.881      0.506

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100       3.5G      1.395      1.507     0.0189         12        640: 100% ━━━━━━━━━━━━ 482/482 5.0it/s 1:36
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.6it/s 2.4s
                   all        550        557      0.742      0.784      0.862      0.486

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100       3.5G      1.422       1.26    0.01932          7        640: 100% ━━━━━━━━━━━━ 482/482 5.1it/s 1:35
                 Class     Images  Instances      Box(P 

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [7]:
import lapconv_backbone  # must be imported before loading a LapConv checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,380,102 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1145.3±310.7 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.3Kit/s 0.4s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 13.3it/s 2.6s
                   all        550        552      0.974      0.937      0.965      0.582
Speed: 0.6ms preprocess, 2.5ms inference, 0.0ms loss, 0.1ms postprocess per image


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,LapConv Backbone (Paper 27),2.38,5.34,0.974,0.937,0.955,0.965


## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper27_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_lapconv_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_lapconv.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `lapconv_backbone.py` | Jetson (needed to load the LapConv `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper27_backbone_edge_bench.zip paper27_backbone_edge_bench`.

In [8]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper27_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("lapconv_backbone.py", EDGE / "lapconv_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import lapconv_backbone  # needed to load the LapConv checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper27_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper27_backbone_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 146 layers, 2,504,835 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_lapconv_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (5.1 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 216ms
 Downloaded ncnn
Prepared 1 package in 174ms
Installed 1 package in 3ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements:

pnnxparam = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = lapconv_backbone.LapConv
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bot

NCNN: export success ✅ 6.9s, saved as '/kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model' (9.3 MB)

Export complete (7.4s)
Results saved to /kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_lapconv_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
bb_lapconv   -> bb_lapconv.pt, bb_lapconv_ncnn_model/

Download paper27_backbone_edge_bench.zip from the Output tab.


### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 27 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [9]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_lapconv": None}
FPS_JETSON = {"bb_lapconv": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
LapConv Backbone (Paper 27),2.380000,5.340000,0.974000,0.937000,0.955000,0.965000,--,--


## 8. Zip all results

Run this last. It creates **`paper27_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper27_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `lapconv_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_lapconv_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [10]:
import zipfile

ALL_ZIP = WORK / "paper27_backbone_all_results.zip"
folders = [WORK / "paper27_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper27_backbone_edge_bench/   59 files
runs/                          26 files
weights/bb_lapconv_best.pt

Download paper27_backbone_all_results.zip (44.3 MB) from the Output tab.
